# 03 · Analyze, reconcile and save the handoff

Recalculate the saved fact, independently calculate the same 22 metrics from the raw CSV, and export full/sample controls for SQL, Excel and Power BI. Keep the exact values; format percentages only for display. This notebook contains all its own code.

In [ ]:
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
assert (ROOT / "sql" / "01_setup.sql").is_file(), "Open Jupyter from the project root."
DATA = ROOT / "notebooks" / "data"
RAW = DATA / "raw"
OUT = DATA / "processed"
EXPORT = DATA / "exports"
QUALITY = DATA / "quality"
for folder in (OUT, EXPORT, QUALITY):
    folder.mkdir(parents=True, exist_ok=True)

import pyarrow.parquet as pq
import numpy as np
import matplotlib.pyplot as plt

CAUSES = ['air_system_delay_minutes', 'security_delay_minutes', 'airline_delay_minutes',
          'late_aircraft_delay_minutes', 'weather_delay_minutes']


## Define metrics and additive components

Eligible arrivals have cancelled = 0, diverted = 0 and a recorded arrival delay. The custom boundary remains ≤15 / >15 / >60 minutes. An empty eligible population has unavailable rates (NaN), not a measured zero. The reliability index is an optional project score.

In [ ]:
def components(frame):
    arrived = frame["cancelled"].eq(0) & frame["diverted"].eq(0) & frame["arrival_delay_minutes"].notna()
    delay = frame.loc[arrived, "arrival_delay_minutes"]
    return {
        "scheduled_flights": len(frame),
        "cancelled_flights": int(frame["cancelled"].eq(1).sum()),
        "diverted_flights": int(frame["diverted"].eq(1).sum()),
        "arrived_flights": int(arrived.sum()),
        "on_time_arrivals": int(delay.le(15).sum()),
        "delay_over_15_flights": int(delay.gt(15).sum()),
        "severe_delay_flights": int(delay.gt(60).sum()),
        "arrival_delay_minutes": float(delay.sum()),
        "arrival_delay_non_null_flights": len(delay),
        **{field: float(frame[field].sum()) for field in CAUSES},
    }

def rates(total):
    result = dict(total)
    arrived, scheduled = result["arrived_flights"], result["scheduled_flights"]
    def divide(numerator, denominator):
        return numerator / denominator if denominator else float("nan")
    for name, numerator, denominator in [
        ("on_time_arrival_rate", "on_time_arrivals", arrived),
        ("average_arrival_delay_minutes", "arrival_delay_minutes", arrived),
        ("delay_over_15_rate", "delay_over_15_flights", arrived),
        ("severe_delay_rate", "severe_delay_flights", arrived),
        ("cancellation_rate", "cancelled_flights", scheduled),
        ("diversion_rate", "diverted_flights", scheduled),
    ]:
        result[name] = divide(result[numerator], denominator)
    result["route_reliability_index"] = (
        max(0, 100 - 50 * result["delay_over_15_rate"] - 50 * result["cancellation_rate"])
        if arrived and scheduled else float("nan")
    )
    return result

fact = pq.ParquetFile(OUT / "fact_flight.parquet")
totals = None
delay_values = []
segments = {name: [] for name in ["airline_code", "route_code", "month", "departure_hour", "peak_period"]}
filter_totals = {}
filters = {
    "all": lambda f: pd.Series(True, index=f.index),
    "AA": lambda f: f["airline_code"].eq("AA"),
    "January": lambda f: f["flight_date"].dt.month.eq(1),
    "JFK-LAX": lambda f: f["route_code"].eq("JFK-LAX"),
    "cancelled_only": lambda f: f["cancelled"].eq(1),
}
for batch in fact.iter_batches(batch_size=250_000):
    frame = batch.to_pandas()
    values = pd.Series(components(frame), dtype="float64")
    totals = values if totals is None else totals.add(values)
    eligible = frame["cancelled"].eq(0) & frame["diverted"].eq(0) & frame["arrival_delay_minutes"].notna()
    delay_values.append(frame.loc[eligible, "arrival_delay_minutes"].to_numpy())
    for name, predicate in filters.items():
        part = pd.Series(components(frame.loc[predicate(frame)]), dtype="float64")
        filter_totals[name] = part if name not in filter_totals else filter_totals[name].add(part)
    enriched = frame.assign(
        month=frame["flight_date"].dt.month, arrived=eligible,
        late=eligible & frame["arrival_delay_minutes"].gt(15),
        severe=eligible & frame["arrival_delay_minutes"].gt(60),
        eligible_delay=frame["arrival_delay_minutes"].where(eligible, 0),
    )
    for field in segments:
        segments[field].append(enriched.groupby(field, dropna=False).agg(
            scheduled_flights=("source_row_number", "size"),
            cancelled_flights=("cancelled", "sum"), diverted_flights=("diverted", "sum"),
            arrived_flights=("arrived", "sum"), late_flights=("late", "sum"),
            severe_delay_flights=("severe", "sum"), arrival_delay_minutes=("eligible_delay", "sum"),
        ))
python_metrics = rates(totals.to_dict())
python_metrics["median_arrival_delay_minutes"] = float(np.median(np.concatenate(delay_values)))
del delay_values
pd.DataFrame([python_metrics]).to_csv(EXPORT / "python_kpi_summary.csv", index=False)
display(pd.Series(python_metrics).to_frame("value"))


## Check the raw source independently

This code works directly with uppercase source fields and does not call the transformation or `components`. It checks preservation of source metrics across the saved handoff. The median uses all eligible values together, never the average of chunk medians.

In [ ]:
raw_totals = {name: 0.0 for name in totals.index}
raw_delay_values = []
cause_sources = ["AIR_SYSTEM_DELAY", "SECURITY_DELAY", "AIRLINE_DELAY", "LATE_AIRCRAFT_DELAY", "WEATHER_DELAY"]
for raw in pd.read_csv(RAW / "flights.csv", chunksize=250_000, low_memory=False):
    eligible = raw["CANCELLED"].eq(0) & raw["DIVERTED"].eq(0) & raw["ARRIVAL_DELAY"].notna()
    delay = raw.loc[eligible, "ARRIVAL_DELAY"]
    raw_totals["scheduled_flights"] += len(raw)
    raw_totals["cancelled_flights"] += int(raw["CANCELLED"].eq(1).sum())
    raw_totals["diverted_flights"] += int(raw["DIVERTED"].eq(1).sum())
    raw_totals["arrived_flights"] += len(delay)
    raw_totals["arrival_delay_non_null_flights"] += len(delay)
    raw_totals["on_time_arrivals"] += int((delay <= 15).sum())
    raw_totals["delay_over_15_flights"] += int((delay > 15).sum())
    raw_totals["severe_delay_flights"] += int((delay > 60).sum())
    raw_totals["arrival_delay_minutes"] += float(delay.sum())
    for original, target in zip(cause_sources, CAUSES):
        raw_totals[target] += float(raw[original].sum())
    raw_delay_values.append(delay.to_numpy())

r = raw_totals
a, s = r["arrived_flights"], r["scheduled_flights"]
r.update(on_time_arrival_rate=r["on_time_arrivals"]/a,
         average_arrival_delay_minutes=r["arrival_delay_minutes"]/a,
         delay_over_15_rate=r["delay_over_15_flights"]/a,
         severe_delay_rate=r["severe_delay_flights"]/a,
         cancellation_rate=r["cancelled_flights"]/s,
         diversion_rate=r["diverted_flights"]/s)
r["route_reliability_index"] = max(0, 100 - 50*r["delay_over_15_rate"] - 50*r["cancellation_rate"])
r["median_arrival_delay_minutes"] = float(np.median(np.concatenate(raw_delay_values)))
del raw_delay_values
recon = pd.DataFrame({"metric": list(python_metrics), "python_value": list(python_metrics.values()),
                      "raw_value": [r[name] for name in python_metrics]})
recon["variance"] = recon["python_value"] - recon["raw_value"]
# Source components/counts and integer-minute sums must be exact.
recon["tolerance"] = [0 if name in totals.index else 1e-10 for name in recon["metric"]]
recon["status"] = np.where(recon["variance"].abs() <= recon["tolerance"], "PASS", "FAIL")
recon.to_csv(QUALITY / "reconciliation.csv", index=False)
assert len(recon) == 22 and recon["status"].eq("PASS").all(), "Raw/fact reconciliation failed"
display(recon)


## Compare segments and see the pattern

Use sums of counts to calculate each rate. Carrier screening is 10,000 scheduled flights; route screening is 1,000 for full-year investigation lists. Numeric route labels stay separate from IATA labels. They are incomplete representations of physical routes, so check `airport_code_profile.csv` before drawing a full-year route conclusion.

In [ ]:
summaries = {}
for field, parts in segments.items():
    result = pd.concat(parts).groupby(level=0).sum()
    denominator = result["arrived_flights"].replace(0, np.nan)
    result["on_time_arrival_rate"] = 1 - result["late_flights"] / denominator
    result["delay_over_15_rate"] = result["late_flights"] / denominator
    result["severe_delay_rate"] = result["severe_delay_flights"] / denominator
    result["cancellation_rate"] = result["cancelled_flights"] / result["scheduled_flights"]
    result["average_arrival_delay_minutes"] = result["arrival_delay_minutes"] / denominator
    result["route_reliability_index"] = (
        100 - 50 * result["delay_over_15_rate"] - 50 * result["cancellation_rate"]
    ).clip(lower=0)
    result = result.reset_index()
    summaries[field] = result
    label = {"airline_code": "carrier", "route_code": "route"}.get(field, field)
    result.to_csv(EXPORT / f"mart_{label}_performance.csv", index=False)

carrier = summaries["airline_code"]
route = summaries["route_code"]
display(carrier.sort_values("delay_over_15_rate", ascending=False))
display(route.loc[route["scheduled_flights"] >= 1000].sort_values("route_reliability_index").head(10))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
carrier.sort_values("on_time_arrival_rate").plot.barh(
    x="airline_code", y="on_time_arrival_rate", ax=axes[0], legend=False, color="#245c78")
axes[0].set(title="Arrivals within 15 minutes by carrier", xlabel="Fraction of eligible arrivals", ylabel="Carrier")
summaries["month"].plot.line(x="month", y="on_time_arrival_rate", ax=axes[1], marker="o", legend=False, color="#245c78")
axes[1].set(title="Monthly arrival performance", xlabel="Month in 2015", ylabel="Fraction of eligible arrivals")
fig.tight_layout()
plt.show()


## Save Excel inputs and report controls

SQL independently recomputes both full and sample metrics. CSV files preserve exact baselines for your manual Excel workbook. The existing workbook is not overwritten. Filter controls will also validate the final Power BI cards.

In [ ]:
sample = pd.read_csv(OUT / "validation_sample.csv", dtype={
    "airline_code": "string", "flight_number": "string", "tail_number": "string",
    "origin_airport_code": "string", "destination_airport_code": "string",
    "scheduled_departure_hhmm": "string",
})
sample_metrics = rates(components(sample))
sample_delay = sample.loc[sample["cancelled"].eq(0) & sample["diverted"].eq(0), "arrival_delay_minutes"].dropna()
sample_metrics["median_arrival_delay_minutes"] = float(sample_delay.median())
pd.DataFrame([sample_metrics]).to_csv(EXPORT / "python_sample_kpi_summary.csv", index=False)
pd.DataFrame({"metric": list(sample_metrics), "value": list(sample_metrics.values())}).to_csv(
    EXPORT / "python_sample_kpis_long.csv", index=False)
exceptions = sample.loc[
    sample["cancelled"].eq(1) | sample["diverted"].eq(1) | sample["arrival_delay_minutes"].gt(60),
    ["source_row_number", "flight_date", "airline_code", "route_code", "arrival_delay_minutes", "cancelled", "diverted"],
].copy()
exceptions["investigation_note"] = ""
exceptions.to_csv(EXPORT / "excel_sample_exceptions.csv", index=False)
checks = pd.DataFrame([{"scope": scope, **rates(value.to_dict())} for scope, value in filter_totals.items()])
checks.to_csv(EXPORT / "python_report_checks.csv", index=False)
display(checks[["scope", "scheduled_flights", "arrived_flights", "on_time_arrival_rate", "cancellation_rate"]])
print("Python complete. Next: SQL analysis. See the project README.md.")


## Your interpretation

Write three observations in your own words: one carrier comparison, one time pattern, and one route worth investigating. Include counts and denominators. The index ranks an investigation priority; it does not measure realized improvements. Recorded cause minutes do not prove causation. Explain the numeric-airport-code limitation before presenting full-year route rankings.

Next: run the SQL analysis with `sql/run_sql.py`, then review the Excel workbook and Power BI report. See the project README for the reproduction workflow.